In [44]:
from threading import Semaphore

from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
import numpy as np
from ArxivSearch import ArxivSearch
import re
from sentence_transformers import CrossEncoder



In [4]:
def load_model():
    model_name = "Qwen/Qwen3-8B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True #swith it off to download from hugging face if you don't have it locally
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return {"model": model, "tokenizer": tokenizer}


In [5]:
def generate_arxive_query(model, tokenizer, prompt):
    system_prompt = f"Generate a search query for arxive extracting the key words out of this question: {prompt}.  Return the search query only"
    messages = [
        {"role": "user", "content": system_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    print("Input tokens:", model_inputs["input_ids"].shape[1])
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens= 50
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    answer = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )

    return answer


In [6]:
model_and_tokenizer = load_model()
model = model_and_tokenizer["model"]
tokenizer = model_and_tokenizer["tokenizer"]

Loading weights:   0%|          | 0/399 [00:02<?, ?it/s]

In [7]:
prompt = "How to evaluate RAG system performance."
answer = generate_arxive_query(model, tokenizer, prompt)

Input tokens: 43


In [8]:
display(Markdown(answer))

"evaluating RAG system performance"

In [9]:
arxiv = ArxivSearch()

titles_to_urls = arxiv.search_titles("RAG evaluation", limit=5)

titles_to_urls



{'Mem++: Non-Destructive Memory for Long-Term Organizational LLM Agents': 'https://arxiv.org/pdf/2610.02002',
 'Mapping the RAG Landscape: A Four Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning': 'https://arxiv.org/pdf/2610.01936',
 'A Matryoshka Hierarchical RAG for Efficient Multi-Hop Question Answering': 'https://arxiv.org/pdf/2610.01767',
 'AGO AI Quality Gate: Evidence-First Release Decisions for Retrieval-Augmented Generation': 'https://arxiv.org/pdf/2610.01218',
 'LawCompass: Navigating from Legal QA to Multi-Agent Deep Research with Grounded Evidence': 'https://arxiv.org/pdf/2610.01027'}

In [86]:
titles = list(titles_to_urls.keys())


In [11]:

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [12]:
ranks = cross_encoder.rank(prompt, titles)
for rank in ranks:
    print(f"{rank['score']:.2f}\t{titles[rank['corpus_id']]}")

-3.31	Mapping the RAG Landscape: A Four Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning
-5.71	A Matryoshka Hierarchical RAG for Efficient Multi-Hop Question Answering
-11.39	LawCompass: Navigating from Legal QA to Multi-Agent Deep Research with Grounded Evidence
-11.43	AGO AI Quality Gate: Evidence-First Release Decisions for Retrieval-Augmented Generation
-11.46	Mem++: Non-Destructive Memory for Long-Term Organizational LLM Agents


In [13]:
best_title = titles[ranks[0]['corpus_id']]
url = titles_to_urls[best_title]
article = arxiv.get_article_text(url)

In [14]:
article

'1\nMapping the RAG Landscape: A Four-Axis Taxonomy\nof Efficiency, Defense, Interactivity, and Reasoning\nMeghana Sunil1†, Shravya V1†, Shravan Venkatraman2, Joe Dhanith P R1∗\nAbstract—Large Language Models (LLMs) have demonstrated remarkable fluency across many tasks but remain limited by their\nstatic, parameter-bound knowledge and their susceptibility to hallucinating information. Retrieval-Augmented Generation (RAG)\naddresses these issues by incorporating external retrieval into the generation process, grounding model outputs in verifiable and\nup-to-date sources. While prior surveys primarily focus on core RAG architectures and standard pipelines, recent research explores\nbroader challenges and capabilities that extend beyond these foundational designs. This survey provides a consolidated and structured\nexamination of contemporary RAG developments, organizing the field into a four-axis taxonomy: improving retrieval efficiency,\nstrengthening robustness and security, supportin

In [59]:
def word_splitter(text):
    text = re.sub("\\s+", " ", text)
    return re.split("\\s", text)  # Split by single whitespace


def get_chunks_fixed_size_with_overlap(article, url, chunk_size, overlap_fraction):
    text_words = word_splitter(article)
    overlap_int = int(chunk_size * overlap_fraction)
    chunks = []
    step = chunk_size - overlap_int
    count = 1
    for i in range(0, len(text_words), step):
        chunk_words = text_words[i:i + chunk_size]
        chunk = {
            "article_url": url,
            "title": best_title,
            "chunk_index": count,
            "text": " ".join(chunk_words)
        }
        count = count + 1
        chunks.append(chunk)
    return chunks

In [60]:
chunks = get_chunks_fixed_size_with_overlap(article, url, 200, 0.2)


In [61]:
chunked_texts = []
for chunk in chunks:
    chunked_texts.append(chunk["text"])



In [62]:
print(chunked_texts)

['1 Mapping the RAG Landscape: A Four-Axis Taxonomy of Efficiency, Defense, Interactivity, and Reasoning Meghana Sunil1†, Shravya V1†, Shravan Venkatraman2, Joe Dhanith P R1∗ Abstract—Large Language Models (LLMs) have demonstrated remarkable fluency across many tasks but remain limited by their static, parameter-bound knowledge and their susceptibility to hallucinating information. Retrieval-Augmented Generation (RAG) addresses these issues by incorporating external retrieval into the generation process, grounding model outputs in verifiable and up-to-date sources. While prior surveys primarily focus on core RAG architectures and standard pipelines, recent research explores broader challenges and capabilities that extend beyond these foundational designs. This survey provides a consolidated and structured examination of contemporary RAG developments, organizing the field into a four-axis taxonomy: improving retrieval efficiency, strengthening robustness and security, supporting user-dr

In [63]:
ranks = cross_encoder.rank(prompt, chunked_texts)
for rank in ranks:
    print(f"{rank['score']:.2f}\t{chunked_texts[rank['corpus_id']]}")

5.12	coordination costs are taken into account. 5.5 Explainability and Interpretability One of the core promises of RAG systems is improved trans- parency over traditional black-box LLMs, owing to their ability to explicitly reference external sources: • By anchoring responses in retrieved content, RAG sys- tems allow users to trace the origins of generated outputs, verify supporting documents, and understand how infor- mation was synthesized [238, 83]. This provenance-aware generation is particularly valuable in regulated domains such as healthcare, finance, and law, where justification and auditability are essential [27, 224, 83], making RAG a natural candidate for applications that require both high performance and clear evidence trails. • Evaluation metrics for RAG explainability have evolved beyond traditional retrieval benchmarks to include con- textual precision and contextual recall, which assess not only relevance but also the semantic fit of retrieved context within the gener

In [87]:
def select_top_k(ranks, k):
    top = []
    for rank in ranks[:k]:
        chunk = chunks[rank["corpus_id"]]
        top.append(chunk)
    return top

In [83]:
def generate_answer(model, tokenizer, docs, input):
    context = ""
    for item in docs:
        context += f"""
        Title: {item['title']}
        Content: {item['text']}
        Url:{item["article_url"]}
        ---
    """

    prompt = f"""
        Answer the question using only the information provided in the context.
        If the context does not contain enough information to answer the question, say:
        "I don't have enough information in the provided context."
        Do not use information that is not supported by the context.
        Context:
        {context}
        Question:
        {input}
        Answer:
    """
    # prompt = f"""
    #     Answer the user's question using ONLY the sources provided below.
    #
    #     Requirements:
    #     - Support factual claims with citations in the format [Source 1], [Source 2].
    #     - Include the source URL after each citation.
    #     - Do not invent citations or URLs.
    #     - Only cite URLs explicitly provided in the sources.
    #     - If the provided sources do not contain enough information to answer,
    #       say that there is insufficient information.
    #
    #     Sources:
    #     {context}
    #
    #     User question:
    #     {input}
    #
    #     Answer:
    #     """
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    print("Input tokens:", model_inputs["input_ids"].shape[1])
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens= 500
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    answer = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )

    return answer



In [88]:
def flow(model, tokenizer, prompt):
    answer = generate_arxive_query(model, tokenizer, prompt)
    arxiv = ArxivSearch()
    titles_to_urls = arxiv.search_titles("RAG evaluation", limit=5)
    titles = list(titles_to_urls.keys())
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")
    ranks = cross_encoder.rank(prompt, titles)
    best_title = titles[ranks[0]['corpus_id']]
    url = titles_to_urls[best_title]
    article = arxiv.get_article_text(url)
    chunks = get_chunks_fixed_size_with_overlap(article, url, 200, 0.2)
    chunked_texts = []
    for chunk in chunks:
        chunked_texts.append(chunk["text"])
    chunk_ranks = cross_encoder.rank(prompt, chunked_texts)
    top = select_top_k(chunk_ranks, 2)
    return generate_answer(model, tokenizer, top, prompt)








In [91]:
prompt = "How to evaluate RAG performance on the system level?"
ans = flow(model, tokenizer, prompt)

Input tokens: 46


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Input tokens: 756


In [92]:
display(Markdown(ans))

To evaluate RAG performance on the system level, one can use standard tools such as Mean Reciprocal Rank (MRR) and Mean Average Precision (MAP) for retrieval diagnostics. Additionally, newer frameworks like FRAMES [21] and Face4RAG [199] introduce hallucination detection and logical consistency scoring for end-to-end system integrity. These methods help ensure the system's overall performance and reliability.

## Evaluation
### System level:
- Token usage per prompt.
- End-to-End Latency
- Context recall (relevant answers / (relevant answers + irrelevant answers). To determine whether an answer is relevant, I will use a threshold of similarity between the actual answer and the correct answer. For example, the cosine similarity is 0.9 then the answer is marked relevant.
- Context precision (relevant answers / total answers)
- Faithfulness (Is the answer strictly true to the retrieved context?" (Checks for hallucination) I am going to use RAGAS library ( if I have no issues configuring it and a LLM-as-judge)
### Retrieval level
- Context precision/ recall @ k
- Mean reciprocal rank (MRR)
- Normalized Discounted Cumulative Gain (NDCG)  how well top-k is sorted




## To do:
- generate ground truth
- test and record the results
- download 1 -2 more models and test the same flow with them. Chose the one that performed best
- With best model, fine-tune the system:
    - test different numbers of searched titles
    - test different number of downloaded articles
    - try different chunk sizes
    - try different number of top k in the chunk ranker
    - try a set of promot augmentation.